In [15]:
import pandas as pd
import numpy as np

In [16]:
# Load database
df_database = pd.read_csv('../data/car_database.csv', sep=';')

In [17]:
# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

# --- AHP CONFIG ---
# Subjective Criteria
ahp_criteria_direction = {
    'cost': 'min', 
    'horsepower': 'max', 
    'transmission': 'max', 
    'city_fuel_economy': 'max', 
    'ground_clearance': 'max', 
    'rear_power_windows': 'max', 
    'power_side_mirrors': 'max', 
    'infotainment_system': 'max', 
    'rear_parking_sensors': 'max', 
    'fog_lights': 'max', 
    'roof_rails': 'max'
}

# Hierarchy Tiers
tier_0_essential    = ['cost']
tier_1_must_have    = ['infotainment_system', 'rear_parking_sensors', 'fog_lights']
tier_2_very_nice    = ['horsepower', 'ground_clearance',  'roof_rails']
tier_3_nice_to_have = ['transmission', 'rear_power_windows', 'power_side_mirrors', 'city_fuel_economy']

# --- GAUSSIAN CONFIG (Objective Criteria - WHITELIST) ---
# Define specifically which columns allow statistical analysis.
# 'cost' is included here to act as a Super-Criterion (Weighted by AHP + Gaussian).
gaussian_candidates = [
    'cost',                 # included for Double Weighting
    'torque',               # performance
    'highway_fuel_economy', # efficiency
    'payload_capacity',     # utility
    'wheelbase'             # often correlates with space/comfort
]

# ==============================================================================
# 2. PRE-PROCESSING
# ==============================================================================

def clean_data(df):
    df_clean = df.copy()
    
    # 1. Text to Number conversions
    if 'transmission' in df_clean.columns:
        df_clean['transmission'] = df_clean['transmission'].apply(lambda x: 1 if str(x).lower().strip() == 'automatic' else 0)
    
    if 'ground_clearance' in df_clean.columns:
        df_clean['ground_clearance'] = df_clean['ground_clearance'].apply(
            lambda x: 2 if str(x).lower().strip() == 'high' else (1 if str(x).lower().strip() == 'medium' else 0)
        )

    # 2. General conversions
    for col in df_clean.columns:
        if df_clean[col].dtype == 'bool':
            df_clean[col] = df_clean[col].astype(int)
        if df_clean[col].dtype == 'object':
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

    if 'car' in df_clean.columns and 'version' in df_clean.columns:
        df_clean['car_version'] = df_clean['car'] + ' - ' + df_clean['version']
        df_clean.set_index('car_version', inplace=True)
    
    df_numeric = df_clean.select_dtypes(include=[np.number])
    return df_numeric, df_clean

df_numeric, df_full = clean_data(df_database)

# ==============================================================================
# 3. AHP MODULE (SUBJECTIVE)
# ==============================================================================

def calculate_ahp_weights(criteria_dict, t0, t1, t2, t3):
    cols = list(criteria_dict.keys())
    n = len(cols)
    if n == 0: return pd.Series()
    
    matrix = np.ones((n, n))
    idx = {name: i for i, name in enumerate(cols)}
    
    # Saaty Scale Reference (From original script)
    # ESSENTIAL is 2x more important than MUST
    STEP_ESSENTIAL_MUST = 2.0
    # MUST is 2x more important than VERYNICE
    STEP_MUST_VERYNICE = 2.0
    # VERYNICE is 2x more important than NICE
    STEP_VERYNICE_NICE = 2.0 
    
    def apply_weight(strong, weak, weight):
        for s in strong:
            for w in weak:
                if s in idx and w in idx:
                    matrix[idx[s], idx[w]] = weight
                    matrix[idx[w], idx[s]] = 1.0 / weight

    # 1. Neighbours Crossing (Directly Connected)
    apply_weight(t0, t1, STEP_ESSENTIAL_MUST)
    apply_weight(t1, t2, STEP_MUST_VERYNICE)
    apply_weight(t2, t3, STEP_VERYNICE_NICE)

    # 2. Neighbours Crossing with two steps of distance
    # ESSENTIAL -> VERYNICE (2 * 2 = 4)
    apply_weight(t0, t2, STEP_ESSENTIAL_MUST * STEP_MUST_VERYNICE)
    # MUST -> NICE (2 * 2 = 4)
    apply_weight(t1, t3, STEP_MUST_VERYNICE * STEP_VERYNICE_NICE)

    # 3. Neighbours Crossing with three steps of distance
    # ESSENTIAL -> NICE (2 * 2 * 2 = 8)
    apply_weight(t0, t3, STEP_ESSENTIAL_MUST * STEP_MUST_VERYNICE * STEP_VERYNICE_NICE)
    
    col_sums = matrix.sum(axis=0)
    weights = (matrix / col_sums).mean(axis=1)
    return pd.Series(weights, index=cols)

weights_ahp_raw = calculate_ahp_weights(
    ahp_criteria_direction, 
    tier_0_essential, tier_1_must_have, tier_2_very_nice, tier_3_nice_to_have
)

print("--- AHP Weights (Subjective) ---")
print(weights_ahp_raw.sort_values(ascending=False))
print("-" * 30)

# ==============================================================================
# 4. GAUSSIAN MODULE (OBJECTIVE - WHITELIST MODE)
# ==============================================================================

def calculate_gaussian_weights(df, whitelist_cols):
    # Filter 1: Column must exist in DataFrame
    available_cols = [c for c in whitelist_cols if c in df.columns]
    
    # Filter 2: Column must have variance > 0
    valid_cols = []
    for c in available_cols:
        if df[c].std() > 0:
            valid_cols.append(c)
            
    if not valid_cols: return pd.Series()

    df_g = df[valid_cols].copy()
    norm_matrix = df_g / df_g.sum()
    
    # Coefficient of Variation (Std / Mean)
    gaussian_factors = norm_matrix.std() / norm_matrix.mean()
    weights = gaussian_factors / gaussian_factors.sum()
    
    return weights

weights_gauss_raw = calculate_gaussian_weights(df_numeric, whitelist_cols=gaussian_candidates)

print("--- Gaussian Weights (Selected Objective Criteria) ---")
print(weights_gauss_raw.sort_values(ascending=False))
print("-" * 30)

# ==============================================================================
# 5. MULTI CRITERIA DECISION ANALYSIS ENGINE
# ==============================================================================

def execute_mcda(df, w_ahp, w_gauss, ahp_directions):
    
    # 1. Unified Normalization
    df_norm = pd.DataFrame(index=df.index)
    all_criteria = list(set(list(w_ahp.index) + list(w_gauss.index)))
    
    for col in all_criteria:
        direction = ahp_directions.get(col, 'max') 
        vals = df[col]
        
        if vals.sum() == 0:
            df_norm[col] = 0
            continue
            
        if direction == 'min':
            inv = 1 / (vals + 1e-9)
            df_norm[col] = inv / inv.sum()
        else:
            df_norm[col] = vals / vals.sum()

    # 2. Test Run (Calculate independent scores)
    cols_ahp = [c for c in w_ahp.index if c in df_norm.columns]
    cols_gauss = [c for c in w_gauss.index if c in df_norm.columns]
    
    score_ahp_only = df_norm[cols_ahp].dot(w_ahp[cols_ahp])
    score_gauss_only = df_norm[cols_gauss].dot(w_gauss[cols_gauss])
    
    # 3. Measure Dispersion (Standard Deviation)
    std_ahp = score_ahp_only.std()
    std_gauss = score_gauss_only.std()
    
    total_variation = std_ahp + std_gauss
    
    if total_variation == 0:
        smart_ahp_share = 0.5
    else:
        # Higher variation = Higher weight
        smart_ahp_share = std_ahp / total_variation
        
    print(f"\n--- Smart Weighting Calculated ---")
    print(f"AHP StdDev: {std_ahp:.4f} | Gaussian StdDev: {std_gauss:.4f}")
    print(f"-> AHP Share: {smart_ahp_share*100:.2f}% | Gaussian Share: {(1-smart_ahp_share)*100:.2f}%")
    print("-" * 30)

    # 4. Final Weight Composition
    final_weights = {}
    for col in all_criteria:
        w_a = w_ahp.get(col, 0)
        w_g = w_gauss.get(col, 0)
        
        # Combine weights
        final_weights[col] = (w_a * smart_ahp_share) + (w_g * (1 - smart_ahp_share))
        
    ser_final_weights = pd.Series(final_weights)
    
    # 5. Final Score
    df_results = df.copy()
    common_cols = [c for c in ser_final_weights.index if c in df_norm.columns]
    df_results['Score_Final'] = df_norm[common_cols].dot(ser_final_weights[common_cols])
    
    return df_results.sort_values('Score_Final', ascending=False), ser_final_weights

# Execute
ranking, weights_used = execute_mcda(
    df_numeric, 
    weights_ahp_raw, 
    weights_gauss_raw, 
    ahp_criteria_direction
)

# ==============================================================================
# 6. REPORT
# ==============================================================================

print("\n=== FINAL MCDA RANKING ===")
# Display Cost, Horsepower (AHP), Torque (Gaussian) and Final Score
cols_view = ['cost', 'horsepower', 'torque', 'Score_Final'] 
cols_view = [c for c in cols_view if c in ranking.columns]
print(ranking[cols_view].head(10))

print("\n=== TOP INFLUENCERS ===")
print(weights_used.sort_values(ascending=False).head(10))

--- AHP Weights (Subjective) ---
cost                    0.266667
fog_lights              0.133333
rear_parking_sensors    0.133333
infotainment_system     0.133333
horsepower              0.066667
roof_rails              0.066667
ground_clearance        0.066667
transmission            0.033333
power_side_mirrors      0.033333
city_fuel_economy       0.033333
rear_power_windows      0.033333
dtype: float64
------------------------------
--- Gaussian Weights (Selected Objective Criteria) ---
torque                  0.575869
cost                    0.207968
highway_fuel_economy    0.157931
payload_capacity        0.058232
dtype: float64
------------------------------

--- Smart Weighting Calculated ---
AHP StdDev: 0.0138 | Gaussian StdDev: 0.0045
-> AHP Share: 75.42% | Gaussian Share: 24.58%
------------------------------

=== FINAL MCDA RANKING ===
                                                         cost  horsepower  \
car_version                                                   